In [1]:
import pandas as pd
import pymysql
from typing import Union, List

def fetch_unique_indicators(
    db_info: dict,
    table_name: str = "Korea_company_valuation_ver2"
) -> list:
    """
    DB 테이블에서 indicator 컬럼의 unique 값 리스트 반환
    """

    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info["port"],
        user=db_info["user"],
        password=db_info["password"],
        database=db_info["database"],
        charset="utf8mb4"
    )

    try:
        sql = f"""
        SELECT DISTINCT indicator
        FROM {table_name}
        ORDER BY indicator
        """
        df = pd.read_sql(sql, conn)
    finally:
        conn.close()

    return df["indicator"].tolist()

def fetch_indicator_pivot(
    db_info: dict,
    indicator: Union[str, List[str]],
    forecast_date: str,
    ticker: str,
    table_name: str = "Korea_company_valuation_ver2"
) -> pd.DataFrame:
    """
    Parameters
    ----------
    db_info : dict
    indicator : str or list[str]
        예: "psr_ETS" 또는 ["psr_ETS", "psr_SARIMA"]
    forecast_date : str
        예: "2025-12-05"
    ticker : str
        예: "A005930"
    table_name : str

    Returns
    -------
    DataFrame
        index   : date
        columns : indicator
        values  : value
    """

    # indicator를 리스트로 통일
    if isinstance(indicator, str):
        indicator = [indicator]

    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info["port"],
        user=db_info["user"],
        password=db_info["password"],
        database=db_info["database"],
        charset="utf8mb4"
    )

    try:
        indicator_placeholders = ",".join(["%s"] * len(indicator))

        sql = f"""
        SELECT
            date,
            indicator,
            value
        FROM {table_name}
        WHERE forecast_date = %s
          AND ticker = %s
          AND indicator IN ({indicator_placeholders})
        ORDER BY date
        """

        params = [forecast_date, ticker] + indicator
        df = pd.read_sql(sql, conn, params=params)

    finally:
        conn.close()

    if df.empty:
        return pd.DataFrame()

    pivot_df = (
        df.pivot_table(
            index="date",
            columns="indicator",
            values="value",
            aggfunc="last"
        )
        .sort_index()
    )

    return pivot_df



def get_valuation_pivot_by_forecast_date(
    db_info: dict,
    forecast_date: str,
    indicator: str,
    table_name: str = "Korea_company_valuation_ver2"
) -> pd.DataFrame:
    """
    forecast_date + indicator를 기준으로
    index=date, columns=ticker, values=value 형태의 pivot DataFrame 생성
    """

    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info.get("port", 3306),
        user=db_info["user"],
        password=db_info["password"],
        database=db_info["database"],
        charset="utf8mb4"
    )

    try:
        sql = f"""
        SELECT
            date,
            ticker,
            value
        FROM {table_name}
        WHERE forecast_date = %s
          AND indicator = %s
        ORDER BY date, ticker
        """
        df = pd.read_sql(sql, conn, params=[forecast_date, indicator])
    finally:
        conn.close()

    if df.empty:
        raise ValueError(
            f"No data found for forecast_date={forecast_date}, indicator={indicator}"
        )

    pivot_df = (
        df.pivot(index="date", columns="ticker", values="value")
          .sort_index()
    )

    return pivot_df


def _to_numeric_df(df: pd.DataFrame) -> pd.DataFrame:
    """DF 값 전체를 숫자로 강제 변환 (쉼표 포함 문자열 처리)."""
    out = df.copy()
    out = out.applymap(lambda x: str(x).replace(",", "").strip() if isinstance(x, str) else x)
    out = out.apply(pd.to_numeric, errors="coerce")
    return out

def _keep_quarter_end_rows(df: pd.DataFrame) -> pd.DataFrame:
    """
    분기말(3/6/9/12월) + 월말인 날짜만 남김.
    (예: 2025-03-31, 2025-06-30, 2025-09-30, 2025-12-31)
    """
    idx = df.index
    is_q_month = idx.month.isin([3, 6, 9, 12])
    is_month_end = idx.is_month_end
    return df.loc[is_q_month & is_month_end].copy()

def load_indicator_pivot(
    db_info: dict,
    indicator: str,
    forecast_date: str,
    table_name: str = "Korea_company_valuation_ver2",
    agg: str = "last"  # "first", "last", "mean", "sum" 등 사용 가능
) -> pd.DataFrame:
    """
    DB에서 indicator + forecast_date에 해당하는 데이터를 불러와
    (date, ticker) 중복을 집계한 후,
    date를 index, ticker를 column, value를 value로 하는 pivot DataFrame 생성
    """

    query = f"""
        SELECT date, ticker, value
        FROM {table_name}
        WHERE indicator = %s
          AND forecast_date = %s
        ORDER BY date, ticker
    """

    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info.get("port", 3306),
        user=db_info["user"],
        password=db_info["password"],
        database=db_info["database"],
        charset="utf8mb4"
    )

    try:
        df = pd.read_sql(query, conn, params=[indicator, forecast_date])
    finally:
        conn.close()

    if df.empty:
        raise ValueError("조회된 데이터가 없습니다. indicator/forecast_date를 확인해주세요.")

    # (date, ticker) 중복 처리
    if agg in ("first", "last"):
        # 정렬 후 첫/마지막 값 사용
        df = df.sort_values(["date", "ticker"])
        grouped = df.groupby(["date", "ticker"], as_index=False)["value"].agg(agg)
    else:
        # 그 외 agg 함수(mean, sum 등)
        grouped = df.groupby(["date", "ticker"], as_index=False)["value"].agg(agg)

    # pivot
    pivot_df = grouped.pivot(index="date", columns="ticker", values="value")

    # index를 datetime으로 확실히 변환
    pivot_df.index = pd.to_datetime(pivot_df.index)

    return pivot_df


def calc_6month_growth_rate(pivot_df: pd.DataFrame) -> pd.DataFrame:
    df = pivot_df.copy()
    df.index = pd.to_datetime(df.index)

    # 숫자형 변환 (문자열로 들어왔을 경우 필수)
    df = df.apply(pd.to_numeric, errors='coerce')

    # ---- 6개월 전 대비 증감률 계산 ----
    # (현재값 - 6개월전값) / 6개월전값
    growth_6m = (df - df.shift(6)) / df.shift(6)

    # 마지막 시점 기준의 증감률만 추출 (가장 최근 row)
    latest_growth = growth_6m.iloc[-1].dropna()

    # 내림차순 정렬
    ranking_df = latest_growth.sort_values(ascending=False).reset_index()
    ranking_df.columns = ['ticker', 'growth_rate_6m']

    # Rank 부여
    ranking_df.insert(0, 'rank', range(1, len(ranking_df)+1))

    return ranking_df, growth_6m   # ranking + 전체 growth matrix도 반환



In [2]:
from DATA.stock_invest_function import get_db_host

db_info = {
    "host": get_db_host(),
    "port": 3307,
    "user": "stox7412",
    "password": "Apt106503!~",
    "database": "investar",
}

indicators = fetch_unique_indicators(
    db_info=db_info,
    table_name="Korea_company_valuation_ver2"
)

print(indicators)


['ensemble_forecast', 'ensemble_valuation', 'exog_var', 'exp_smoothing_forecast', 'forecast_date', 'is_forecast', 'lstm_forecast', 'lstm_valuation', 'matched_ttm_without_exog', 'matched_ttm_with_exog', 'mc_ets', 'mc_lstm', 'mc_prophet', 'mc_sarima_exog', 'mc_sarima_noexog', 'mc_theta', 'prophet_forecast', 'prophet_valuation', 'psr', 'psr_ETS', 'psr_LSTM', 'psr_Prophet', 'psr_SARIMA_exog', 'psr_SARIMA_noexog', 'psr_Theta', 'revenue_ensemble_forecast', 'revenue_ets', 'revenue_ets_ttm', 'revenue_exp_smoothing_forecast', 'revenue_lstm', 'revenue_lstm_forecast', 'revenue_lstm_ttm', 'revenue_prophet', 'revenue_prophet_forecast', 'revenue_prophet_ttm', 'revenue_sarima', 'revenue_sarima_exog', 'revenue_sarima_exog_ttm', 'revenue_sarima_ttm', 'revenue_theta', 'revenue_theta_ttm', 'revenue_without_exog_forecast', 'revenue_with_exog_forecast', 'sarima_forecast', 'sarima_valuation', 'ttm_revenue', 'ttm_revenue_without_exog', 'ttm_revenue_with_exog', 'valuation']


In [3]:
pivot_df = fetch_indicator_pivot(
    db_info=db_info,
    indicator= "revenue_sarima",
    forecast_date= "2026-01-23",
    ticker="A140860"
)

print(pivot_df)

indicator      revenue_sarima
date                         
2004-12-31                  0
2005-12-31                  0
2006-12-31                  0
2007-12-31                  0
2008-12-31                  0
...                       ...
2026-12-31  74606819923.94588
2027-03-31  58530762450.00469
2027-06-30  65480464194.32125
2027-09-30  62972269627.43147
2027-12-31  81782419255.98618

[64 rows x 1 columns]


#### 매출증가율 순위 랭킹 부여

In [4]:
revenue_df = load_indicator_pivot(
    db_info=db_info,
    indicator="revenue_sarima",
    forecast_date="2026-01-25",
    agg="last"   # 필요하면 "mean" 등으로 변경 가능
)

In [5]:
revenue_df.tail(24)

ticker,A004000,A059210,A189300,A214150
date,,,,
2023-12-31,411264998330,21320944080,86279646460,47004803940
2024-03-31,399444149820,22502545390,46697934210,50379744700
2024-06-30,422072382130,25327086210,71699403380,58742159580
2024-09-30,420414721170,23149639630,62360862090,59413228850
2024-12-31,428613936760,23056779800,77011249200,74403832250
2025-03-31,445589676650,27700050590,43350439170,77113296530
2025-06-30,424673817980,25474754590,74736003380,83283805950
2025-09-30,443363087560,23534544310,77226483700,82997291550
2025-10-31,NaN,NaN,NaN,86942000493.14764


#### 매출액 증가 순위 추출

In [15]:
# ---------------- 사용 예시 ---------------- #
ranking_df, full_growth_matrix = calc_6month_growth_rate(revenue_df)
ranking_df.head(20)

,rank,ticker,growth_rate_6m
0,1,A000500,0.335277
1,2,A005930,0.171382
2,3,A052400,0.101283
3,4,A004000,0.072221


In [9]:
pivot_df = load_indicator_pivot(
    db_info=db_info,
    indicator="mc_sarima_noexog",
    forecast_date="2026-01-23",
    agg="last"   # 필요하면 "mean" 등으로 변경 가능
)


#### 시장총액 증가율 예상

In [10]:
# ---------------- 사용 예시 ---------------- #
ranking_df, full_growth_matrix = calc_6month_growth_rate(pivot_df)

print(ranking_df.head(20))

    rank   ticker  growth_rate_6m
0      1  A298040        4.013199
1      2  A066970        0.362928
2      3  A042660        0.315014
3      4  A103590        0.291490
4      5  A140860        0.259363
5      6  A077360        0.196735
6      7  A028300        0.187674
7      8  A010140        0.172480
8      9  A000660        0.172011
9     10  A059090        0.148136
10    11  A035900        0.135404
11    12  A064350        0.112924
12    13  A012450        0.108426
13    14  A009540        0.107325
14    15  A003230        0.102111
15    16  A037270        0.085661
16    17  A052400        0.082870
17    18  A033240        0.082207
18    19  A161390        0.074469
19    20  A052690        0.072282


In [107]:
pivot_df['A068270']

date
2026-01-31     51794844751722.74
2026-02-28     52515274542448.42
2026-03-31     56373968873954.08
2026-04-30     57719527539877.58
2026-05-31     51092827091279.27
2026-06-30     55762521730953.27
2026-07-31     59327123138239.92
2026-08-31    55192298068258.664
2026-09-30     52566514590659.11
2026-10-31     51936433607637.82
2026-11-30     49933068013066.85
2026-12-31    52501604628698.164
2027-01-31     50124179157387.98
2027-02-28    47795608616812.305
2027-03-31      52243136160959.8
2027-04-30    52015415670925.914
2027-05-31     46139312669743.45
2027-06-30     49795631908541.58
2027-07-31     54228967839887.04
2027-08-31    49366787866670.055
2027-09-30     47213715368155.27
2027-10-31     46290165578118.41
2027-11-30     44322151509877.02
2027-12-31      46721983819954.6
Name: A068270, dtype: object

In [61]:
def make_quarterly_df(pivot_df: pd.DataFrame) -> pd.DataFrame:
    df = pivot_df.copy()

    # 1) 인덱스를 날짜형으로 변환
    df.index = pd.to_datetime(df.index)

    # 2) 분기별 합산 (원래가 이미 분기 데이터라면, 분기당 1개 값이라 합과 동일)
    quarterly_df = df.resample('Q').sum(min_count=1)

    return quarterly_df

quarterly_df = make_quarterly_df(pivot_df)
# quarterly_data = quarterly_df.loc['2025' : '2026'].dropna(axis=1)

In [62]:
quarterly_df

ticker,A000270,A000500,A000660,A001440,A002350,A003230,A004000,A005380,A005930,A006730,...,A114810,A123700,A131290,A131970,A140860,A161390,A214150,A253590,A298040,A353200
date,,,,,,,,,,,,,,,,,,,,,
2004-03-31,3376147000000,101881795000,1296647066000,373432832000,70504914000,67789237000,196675906000,6207339000000,14413632000000,5808962000,...,None,None,None,None,None,None,None,None,None,None
2004-06-30,3896716000000,109973900000,1683512230000,389869749000,70086643000,67949910000,199240283000,7183273000000,14979452000000,5833847000,...,None,None,None,None,None,None,None,None,None,None
2004-09-30,3385320000000,94487801000,1542349696000,384005892000,75264103000,66013497000,203669056000,6540110000000,14343943000000,5931607000,...,None,None,None,None,None,None,None,None,None,None
2004-12-31,4599559000000,103289793000,1341844008000,463811191000,89056921000,72844171000,194835143000,7541735000000,13895332000000,5947572000,...,None,None,0,None,0,None,None,None,None,None
2005-03-31,3938854000000,111407808000,1284277192000,376205149000,90311354000,77973114000,172972503000,6170228000000,13812185000000,5513823000,...,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-12-31,31376140337668.59,770134267591.3378,26825705654012.59,1042087253322.1116,768996186498.4485,844141742761.515,407744230253.2603,52699350167787.24,89886062558439.64,81723751228.94035,...,56556561711.163536,53195148438.66028,115278585309.5567,77711574420.26947,74475598484.95325,6702024177474.268,121090573522.5574,23554482842.662975,2034063690793.096,309945224657.5477
2027-03-31,31361826746224.11,803797945953.7312,26381194045881.344,1000829185224.1328,779110168299.165,889216313762.1926,440784427694.8431,51619992259138.24,89920337086424.23,77284385545.15439,...,56682916094.48736,52826040330.767944,103774651087.18085,77709843838.96236,59929993242.75424,6947182819485.081,126216943208.99232,25801246156.801506,1782261346699.6445,302227936912.6604
2027-06-30,33253080747314.09,828178928564.4132,27245472238342.293,1050156663306.1597,817829868932.0833,928961844983.2615,455982167222.1588,54429304083863.19,89183979318300.88,83580985124.89574,...,60991996738.2641,51069713337.52046,122160853940.63486,74411545090.57304,66475815781.58304,7299031552187.043,131639641499.66643,27304776366.988655,2042414834925.4993,326316529970.2444


In [63]:
import pandas as pd
import numpy as np

def rank_growth_2026_vs_2025(quarterly_df: pd.DataFrame) -> pd.DataFrame:
    df_q = quarterly_df.copy()
    df_q.index = pd.to_datetime(df_q.index)

    # 🔹 0) 문자열 → 숫자형으로 변환 (핵심!)
    df_q = df_q.apply(pd.to_numeric, errors='coerce')

    # 1) 연간 매출 = 분기 매출 합산
    annual_df = df_q.resample('Y').sum(min_count=1)
    annual_df.index = annual_df.index.year  # 2004-12-31 → 2004

    # 2) 2025년, 2026년 데이터 체크
    if 2025 not in annual_df.index or 2026 not in annual_df.index:
        raise ValueError("annual_df에 2025 또는 2026 데이터가 없습니다.")

    rev_2025 = annual_df.loc[2025]
    rev_2026 = annual_df.loc[2026]

    # 🔹 2-1) 2025 또는 2026이 전부 NaN인 컬럼 제거 (선택적이지만 안전)
    both_years = pd.concat([rev_2025, rev_2026], axis=1)
    both_years.columns = ['rev_2025', 'rev_2026']
    both_years = both_years.dropna(how='all')

    rev_2025 = both_years['rev_2025']
    rev_2026 = both_years['rev_2026']

    # 3) 성장률 = (2026 - 2025) / 2025
    #    2025 매출이 0이거나 NaN이면 제거
    valid_mask = (rev_2025.notna()) & (rev_2026.notna()) & (rev_2025 != 0)
    rev_2025 = rev_2025[valid_mask]
    rev_2026 = rev_2026[valid_mask]

    growth_2026 = (rev_2026 - rev_2025) / rev_2025

    # 4) 내림차순 정렬 + 랭킹
    ranking_df = growth_2026.sort_values(ascending=False).reset_index()
    ranking_df.columns = ['ticker', 'growth_2026_vs_2025']

    ranking_df.insert(0, 'rank', range(1, len(ranking_df) + 1))

    return ranking_df


# 사용 예시
quarterly_df = make_quarterly_df(pivot_df)   # 1번 함수 재사용

quarterly_data = quarterly_df.loc['2025' : '2026'].dropna(axis=1)

ranking_df = rank_growth_2026_vs_2025(quarterly_data)


In [64]:
ranking_df

,rank,ticker,growth_2026_vs_2025
0,1,A066970,1.029482
1,2,A059090,0.410540
2,3,A031980,0.363870
3,4,A012450,0.328059
4,5,A214150,0.327559
...,...,...,...
59,60,A123700,0.010343
60,61,A002350,0.008477
61,62,A073240,0.004079
62,63,A043150,-0.001202


In [56]:
from datetime import datetime
import os

# 저장 경로
path = r"C:\Users\82108\OneDrive\바탕 화면\investment\data\analysis_results"

# 데이터 (예시)
q_data = quarterly_data.T
# 오늘 날짜 생성
today = datetime.today().strftime("%Y%m%d")
# 파일명 생성
file_name = f"revenue_growth_sarima_{today}.xlsx"
# 전체 파일 경로
full_path = os.path.join(path, file_name)
# 저장
q_data.to_excel(full_path, index=True)
print(f"파일 저장 완료: {full_path}")

파일 저장 완료: C:\Users\82108\OneDrive\바탕 화면\investment\data\analysis_results\revenue_growth_sarima_20251226.xlsx


In [52]:
ranking_df

,rank,ticker,growth_2026_vs_2025
0,1,A066970,0.579162
1,2,A025980,0.525452
2,3,A059090,0.522349
3,4,A031980,0.422202
4,5,A032500,0.412110
...,...,...,...
62,63,A123700,0.002631
63,64,A006910,-0.029605
64,65,A011780,-0.039003
65,66,A051910,-0.067038


In [3]:
# import pymysql
#
# conn = pymysql.connect(**db_info)
# cursor = conn.cursor()
#
# try:
#     # 삭제 전 데이터 개수 확인
#     check_sql = """
#     SELECT COUNT(*) as count
#     FROM Korea_company_valuation_ver2
#     WHERE forecast_date = '2025-12-25'
#     """
#     cursor.execute(check_sql)
#     count_before = cursor.fetchone()[0]
#     print(f"삭제 전 데이터 개수: {count_before:,}")
#
#     # 데이터 삭제
#     delete_sql = """
#     DELETE FROM Korea_company_valuation_ver2
#     WHERE forecast_date = '2025-12-26'
#     """
#     cursor.execute(delete_sql)
#     conn.commit()
#
#     deleted_rows = cursor.rowcount
#     print(f"삭제된 데이터 개수: {deleted_rows:,}")
#
#     # 삭제 후 확인
#     cursor.execute(check_sql)
#     count_after = cursor.fetchone()[0]
#     print(f"삭제 후 남은 데이터 개수: {count_after:,}")
#
# except Exception as e:
#     conn.rollback()
#     print(f"오류 발생: {e}")
#
# finally:
#     cursor.close()
#     conn.close()
#
# print("\n삭제 완료")



삭제 전 데이터 개수: 7,390
삭제된 데이터 개수: 49,215
삭제 후 남은 데이터 개수: 7,390

삭제 완료
